In [191]:
import pandas as pd
import regex as re

In [192]:
data = pd.read_csv("data/translation_data_full_duplicates.csv")


In [193]:
data

,teaviku_laad,sihtrühm,pealkiri,autor,originaal_pealkiri,Žanr,tõlkija,lähtekeel,lähtekirjandus,toimetaja,...,ilmumisaasta,füüsiline_kirjeldus,trükikordus,sarjaandmed,arvustatava_tõlkeraamatu_autor,väljaanne,number_leheküljed,sisu,märkused,id
0,Tõlkearvustus,Täiskasvanute ilukirjandus,Natanael. Kulturhistorialine roman. A. Liepe j...,π,NaN,romaanid,"Meomuttel, Jüri 1868-1948",saksa,saksa,NaN,...,1896,NaN,NaN,NaN,"Liepe, Albert",<p>Olevik</p>,"13. veebr., nr 7, lk 161-162",NaN,Idna Illiv (pseud.) = Jüri Meomuttel,71113
1,Tõlkearvustus,Täiskasvanute ilukirjandus,Talu sulane Tõnis. Ühe Schweitsi jutu järele j...,π,NaN,jutud,teadmata,saksa,šveitsi,NaN,...,1897,NaN,NaN,NaN,"Stael von Holstein, Lucia 1857-1929",<p>Olevik</p>,"28. okt., nr 43, lk 961",NaN,NaN,53196
2,Perioodikas ilmunud tõlge,Täiskasvanute ilukirjandus,Miss Portsia,"Zunk, Paul",teadmata,jutud,"Mällow, J.",teadmata,teadmata,NaN,...,1894,NaN,NaN,NaN,NaN,<p>Eesti Postimees : Lisaleht</p>,nr 21-22,NaN,Väljaandes: Paul Zunk'i uudisjutukese järele J...,63698
3,Kogumikus ilmunud tõlge,Täiskasvanute ilukirjandus,Kodu,"Žukovski, Vassili 1783-1852",NaN,luuletused,"Leppik, Jaan 1861-1943",vene,vene,NaN,...,1889,39 lk. 13 cm,NaN,NaN,NaN,NaN,NaN,<p>Ilmunud kogumikus: Eestistatud laulud 2.</p>,Väljaandes: Wene keelest [autor märkimata]; Ee...,58892
4,Perioodikas ilmunud tõlge,Täiskasvanute ilukirjandus,Kolm õde,"Žukovski, Vassili 1783-1852",teadmata,jutud,G. Kewade,saksa,vene,NaN,...,1886,NaN,NaN,NaN,NaN,<p>Meelelahutaja</p>,"31. dets., nr. 52, lk. 409",NaN,Väljaandes: Shukowsky järele: G. Kewade,55755
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52036,Perioodikas ilmunud tõlge,Täiskasvanute ilukirjandus,Peremeheta kohver,"Adamov, Arkadi 1920-1991",Чемодан без хозяина,jutustused,"Jürna, Juhan-Kaspar 1931-1982",vene,vene,NaN,...,19721973,NaN,NaN,NaN,NaN,<p>Noorte Hääl</p>,"30. nov.; 1.-3., 5., 7.-8., 10., 12., 13.-17.,...",NaN,NaN,82166
52037,Perioodikas ilmunud tõlge,Laste- ja noortekirjandus,Nõiatulp : ulmejutt,"Abramov, Sergei 1944-2024",teadmata,jutud,"Maksing, Meta",vene,vene,NaN,...,19801981,NaN,NaN,NaN,NaN,<p>Säde</p>,"12., 16., 26. juuli; 2., 6., 9., 16., 23., 27....",NaN,NaN,78190
52038,Tõlkearvustus,Täiskasvanute ilukirjandus,Zsigmond Móricz: Mudakuld. Romaan. Ungari keel...,A. K.,Sárarany,romaanid,"Murakin, Ants 1892-1975",ungari,ungari,NaN,...,12929,NaN,NaN,Looduse kroonine romaan nr 1,"Móricz, Zsigmond 1879-1942",<p>Päevaleht</p>,"3. veebr., nr 33, lk 8",NaN,NaN,52312
52039,Tõlkeraamat,Täiskasvanute ilukirjandus,Irmgard : hästi põnew romaan,NaN,teadmata,romaanid,"Neumann, Mihkel 1866-1941",teadmata/undetermined,teadmata/undetermined,NaN,...,19231924,23 annet (733 lk.),NaN,NaN,NaN,NaN,NaN,NaN,Kaanel pealkirja täiendandmed: Ülipõnew romaan...,45109


# Goal: make one big table such that every row of translations is represented, and then all the others as well

## Explode all relevant columns at semicolons

In [194]:
data.columns

Index(['teaviku_laad', 'sihtrühm', 'pealkiri', 'autor', 'originaal_pealkiri',
       'Žanr', 'tõlkija', 'lähtekeel', 'lähtekirjandus', 'toimetaja',
       'ees__järelsõna_autor', 'ilmumiskoht', 'kirjastaja', 'ilmumisaasta',
       'füüsiline_kirjeldus', 'trükikordus', 'sarjaandmed',
       'arvustatava_tõlkeraamatu_autor', 'väljaanne', 'number_leheküljed',
       'sisu', 'märkused', 'id'],
      dtype='str')

In [198]:
data.trükikordus.unique()

<StringArray>
[                                                                                         nan,
                                                                                '2. tr. 1865',
                                                                                     '2. tr.',
                                                                    '2. tr. [esmatrükk 1885]',
                                                                                 '2. tr 1885',
                                                                                     '3. tr.',
                                                                                 '2 tr. 1867',
                                                                    '2. tr. [esmatrükk 1890]',
                                                                                   '2. trükk',
                                                            '3. trükk, esimesed 1841 ja 1842',
                                    

In [199]:
semicolon_splittable = ['author',
       'genre', 'translator', 'source_language',
       'source_literature', 'editor', 'fore_afterword_author',
       'publication_location', 'publisher', 'series',
       'author_of_translated_book_under_review_to_be_removed']
semicolon_splittable = ['autor', 
       'Žanr', 'tõlkija', 'lähtekeel', 'lähtekirjandus', 'toimetaja',
       'ees__järelsõna_autor', 'ilmumiskoht', 'kirjastaja', 'sarjaandmed',
       'arvustatava_tõlkeraamatu_autor']
for label in semicolon_splittable:
    data[label] = data[label].str.split(';')
    data = data.explode(label)
    data[label] = data[label].str.strip()

In [200]:
data = data.reset_index(drop=True)

In [201]:
data.columns

Index(['teaviku_laad', 'sihtrühm', 'pealkiri', 'autor', 'originaal_pealkiri',
       'Žanr', 'tõlkija', 'lähtekeel', 'lähtekirjandus', 'toimetaja',
       'ees__järelsõna_autor', 'ilmumiskoht', 'kirjastaja', 'ilmumisaasta',
       'füüsiline_kirjeldus', 'trükikordus', 'sarjaandmed',
       'arvustatava_tõlkeraamatu_autor', 'väljaanne', 'number_leheküljed',
       'sisu', 'märkused', 'id'],
      dtype='str')

In [202]:
data.to_csv("data/exploded_data.csv", index=False)

In [176]:
correct_colindex = ["author", "translator", "editor", "fore_afterword_author", 
                    "title", "title_original", "genre", "source_language", "source_literature",
                   "publisher", "publication_location", "publication_year", "physical_description",
                   "series", "publication_type", "target_audience", "edition", "n_pages", "content",
                   "issue", "notes"]

In [177]:
for col in data.columns:
    if col not in correct_colindex:
        print(f"{col} is not in data.columns")

author_of_translated_book_under_review_to_be_removed is not in data.columns
id is not in data.columns


In [178]:
data = data.drop(["id", "author_of_translated_book_under_review_to_be_removed"], axis=1)

In [179]:
correct_colindex

['author',
 'translator',
 'editor',
 'fore_afterword_author',
 'title',
 'title_original',
 'genre',
 'source_language',
 'source_literature',
 'publisher',
 'publication_location',
 'publication_year',
 'physical_description',
 'series',
 'publication_type',
 'target_audience',
 'edition',
 'n_pages',
 'content',
 'issue',
 'notes']

In [180]:
data = data.loc[:, correct_colindex]